# Pillar 1 — Foundations walkthrough
Runs `etl_pipeline.py`'s actual functions (imported, not duplicated) one step at a time, so every number below is live pipeline output — not a pre-computed file being re-displayed.

Covers Task 1.1 (`load_projects` / `transform_projects`) and Task 1.3 (`load_employees` / `clean_employees`), issue by issue, in the six categories the brief requires: missing values, invalid date formats, implausible dates, numeric out-of-range, logical inconsistencies, status conflicts.

In [13]:
import sys, os
ROOT = os.getcwd()
while not os.path.isdir(os.path.join(ROOT, "datasets")):
    ROOT = os.path.dirname(ROOT)
os.chdir(ROOT)
sys.path.insert(0, os.path.join(ROOT, "solutions", "submissions", "mohammed_faisal", "01_foundations"))

import pandas as pd
pd.set_option("display.width", 200, "display.max_columns", 30)
from etl_pipeline import (
    load_projects, transform_projects,
    load_employees, clean_employees,
    LEVEL_SALARY_BANDS, LEVEL_EXPERIENCE_BANDS, HIRE_DATE_MIN,
    DATA_DIR, RUN_STATS,
)
print("repo root:", ROOT)
print("DATA_DIR :", DATA_DIR)

repo root: c:\Users\mohammed.faisal\OneDrive - Presight AI\Desktop\stackup-engineering-academy_assessment
DATA_DIR : c:\Users\mohammed.faisal\OneDrive - Presight AI\Desktop\stackup-engineering-academy_assessment\datasets


## Task 1.1 — `projects.csv`

### Step 1 — Load the raw file
The `INFO` lines below are the pipeline's own logging (from `load_projects()`), not notebook output.

In [34]:
raw_projects = load_projects(os.path.join(DATA_DIR, "projects.csv"))

2026-09-23 17:02:17,136 | INFO | Loading projects data...
2026-09-23 17:02:17,165 | INFO | Loaded 500 projects (15 columns)


In [36]:
print("Shape (rows, columns):", raw_projects.shape)
raw_projects.head()

Shape (rows, columns): (500, 15)


,project_id,project_name,department,status,start_date,end_date,budget,actual_cost,project_manager_id,priority,region,budget_variance,is_over_budget,duration_days,budget_utilisation_pct
0,PRJ0001,MLOps Platform,Sustainability,Completed,2023-06-13,2023-12-01,1200000.0,907809.0,EMP0126,Medium,Dubai,-292191.0,False,171,75.65
1,PRJ0002,Tax Automation Platform - Q2,Marketing,In Progress,2025-02-28,NaT,800000.0,795986.0,EMP0045,Medium,Abu Dhabi,-4014.0,False,<NA>,99.50
2,PRJ0003,ERP Consolidation Phase 2,Legal,Completed,2023-11-21,2024-12-16,80000.0,56981.0,EMP0405,Medium,Dubai,-23019.0,False,391,71.23
3,PRJ0004,Employee Wellness App - Q1,Marketing,Completed,2024-06-09,2025-08-17,800000.0,707314.0,EMP0270,Medium,Abu Dhabi,-92686.0,False,434,88.41
4,PRJ0005,Mobile Workforce App - Q4,Finance,In Progress,2023-04-25,NaT,2000000.0,1329426.0,EMP0478,High,Ras Al Khaimah,-670574.0,False,<NA>,66.47


### Step 2 — What's missing before we touch anything
`budget` and `actual_cost` both have nulls — this is why `budget_variance` / `budget_utilisation_pct` are computed as `NaN` first, and only zero-filled afterwards in `transform_projects()` (a deliberate order: the derived metrics must not lie about an unknown value).

In [37]:
raw_projects[["budget", "actual_cost"]].isna().sum()

budget         32
actual_cost    60
dtype: int64

### Step 3 — Transform: standardise status, zero-fill nulls, add `risk_level`

In [38]:
projects_clean = transform_projects(raw_projects)
print(projects_clean.shape)
projects_clean[["project_id", "status", "status_category", "budget", "actual_cost",
                "budget_variance", "budget_utilisation_pct", "duration_days",
                "is_over_budget", "risk_level"]].head(8)

2026-09-23 17:02:52,083 | INFO | Transforming projects data...


(500, 17)


,project_id,status,status_category,budget,actual_cost,budget_variance,budget_utilisation_pct,duration_days,is_over_budget,risk_level
0,PRJ0001,Completed,Closed,1200000.0,907809.0,-292191.0,75.65,171,False,Low
1,PRJ0002,In Progress,Active,800000.0,795986.0,-4014.0,99.50,<NA>,False,Medium
2,PRJ0003,Completed,Closed,80000.0,56981.0,-23019.0,71.23,391,False,Low
3,PRJ0004,Completed,Closed,800000.0,707314.0,-92686.0,88.41,434,False,Low
4,PRJ0005,In Progress,Active,2000000.0,1329426.0,-670574.0,66.47,<NA>,False,Medium
5,PRJ0006,In Progress,Active,2000000.0,1438154.0,-561846.0,71.91,<NA>,False,Medium
6,PRJ0007,Completed,Closed,500000.0,437614.0,-62386.0,87.52,470,False,Low
7,PRJ0008,In Progress,Active,200000.0,172385.0,-27615.0,86.19,<NA>,False,Low


### Step 4 — Sanity checks
`status_category` must only ever be Active / Closed / Pending (or the fallback `Unknown` if a status value doesn't match the map), and `risk_level` shouldn't be entirely skewed to one bucket.

In [39]:
print("status -> status_category mapping used:")
projects_clean.groupby(["status", "status_category"]).size()

status -> status_category mapping used:


status       status_category
Completed    Closed             214
In Progress  Active             189
Not Started  Pending             50
On Hold      Pending             47
dtype: int64

In [40]:
projects_clean["risk_level"].value_counts()

risk_level
High      188
Low       167
Medium    145
Name: count, dtype: int64

## Task 1.3 — `employees.csv`

### Step 1 — Load the raw file
`hire_date` is loaded as text on purpose (`dtype={"hire_date": "string"}`) so that sentinel junk values like `-999` survive to be detected in `clean_employees()`, instead of quietly becoming `NaT` at load time where they'd never be counted or logged.

In [20]:
raw_employees = load_employees(os.path.join(DATA_DIR, "employees.csv"))

2026-09-23 17:02:12,715 | INFO | Loading employees data...
2026-09-23 17:02:12,727 | INFO | Employees: 1000 rows x 12 columns
2026-09-23 17:02:12,731 | INFO | Null counts per column: {'email': 10}


In [21]:
print("Shape (rows, columns):", raw_employees.shape)
raw_employees.head()

Shape (rows, columns): (1000, 12)


,employee_id,full_name,email,department,role,level,hire_date,salary,manager_id,region,status,years_experience
0,EMP0001,Anjali Desai,anjali.desai@presight.ai,Operations,Senior Data Engineer,Mid,2019-08-06,17839,EMP0000,Abu Dhabi,Active,6
1,EMP0002,Mohammed Mansour,mohammed.mansour@presight.ai,Operations,Senior Data Engineer,Senior,2014-08-20,33928,EMP0000,Dubai,Active,11
2,EMP0003,Adam Rahman,adam.rahman@presight.ai,Legal,Compliance Analyst,Mid,2021-02-17,18763,EMP0000,Abu Dhabi,Active,4
3,EMP0004,Fatima Al Suwaidi,fatima.al.suwaidi@presight.ai,Procurement,Procurement Analyst,Mid,2024-05-30,20100,EMP0000,Dubai,Active,2
4,EMP0005,Priya Chen,priya.chen@presight.ai,Procurement,Procurement Analyst,Junior,2024-02-21,14370,EMP0000,Abu Dhabi,Active,0


### Step 2 — Run the full cleaning pass
`clean_employees()` finds and fixes issues in one vectorised pass (no `.iterrows()` anywhere) and logs a detection report. The cell below reads that same report back from `RUN_STATS`, which the function updates in place — so this is the pipeline's own count, not a re-derived one.

In [22]:
salary_history = pd.read_csv(os.path.join(DATA_DIR, "employees_salary_history.csv"))
employees_clean = clean_employees(raw_employees, salary_history)
print(employees_clean.shape)

2026-09-23 17:02:12,781 | INFO | Cleaning employees data...
2026-09-23 17:02:12,791 | INFO | [DQ] Missing emails: 10
2026-09-23 17:02:12,818 | INFO | [DQ] Unparseable hire_date: 8 (values: {'-999': 5, '99999-01-01': 3})
2026-09-23 17:02:12,821 | INFO | [DQ] Implausible hire_date (<1990-01-01 or future): 0
2026-09-23 17:02:12,840 | INFO | [FIX] hire_date recovered from salary history: 0 of 8
2026-09-23 17:02:12,842 | INFO | [DQ] Numeric values outside absolute range (salary 10K-100K, experience 0-50): 5
2026-09-23 17:02:12,844 | INFO | [DQ] Negative years_experience: 5
2026-09-23 17:02:12,860 | INFO | [DQ] years_experience outside level band (after fix): 0
2026-09-23 17:02:12,878 | INFO | [DQ] Salary outside band for level: 3 -> [['EMP0356', 'Junior', 76507], ['EMP0867', 'Junior', 66797], ['EMP0900', 'Junior', 71212]]
2026-09-23 17:02:12,887 | INFO | [DQ] Active employees with no valid hire_date after recovery: 8
2026-09-23 17:02:12,900 | INFO | [DQ] manager_id not present in employees:

(1000, 13)


In [23]:
pd.Series(RUN_STATS["detection"], name="rows_affected").rename_axis("issue_category").to_frame()

,rows_affected
issue_category,
Missing values,10
Invalid date formats,8
Implausible dates,0
Numeric out-of-range,5
Logical inconsistencies (salary/experience vs level),3
Status conflicts,8
Referential issues (manager_id),5
Duplicates,0


### Issue 1 — Missing values
10 rows with a blank `email`. Fixed by deriving `first.last@presight.ai` from the employee's name (company convention); on a collision the employee id is appended so every address stays unique. Every other required column (`employee_id`, `full_name`, `department`, `role`, `level`, `status`) has zero blanks in this dataset, so only email needed a fix — see the detection table above.

In [24]:
missing_email = raw_employees["email"].isna() | (raw_employees["email"].astype("string").str.strip() == "")
print("rows affected:", int(missing_email.sum()))
pd.DataFrame({
    "employee_id": raw_employees.loc[missing_email, "employee_id"],
    "full_name": raw_employees.loc[missing_email, "full_name"],
    "raw_email": raw_employees.loc[missing_email, "email"],
    "derived_email": employees_clean.loc[missing_email, "email"],
})

rows affected: 10


,employee_id,full_name,raw_email,derived_email
357,EMP0358,Anika Malik,NaN,anika.malik@presight.ai
372,EMP0373,Hamad Saleh,NaN,hamad.saleh@presight.ai
460,EMP0461,Mohammed Al Marri,NaN,mohammed.al.marri@presight.ai
535,EMP0536,Nina Laurent,NaN,nina.laurent@presight.ai
646,EMP0647,Salma Al Shamsi,NaN,salma.al.shamsi.emp0647@presight.ai
657,EMP0658,Saif Al Mazrouei,NaN,saif.al.mazrouei.emp0658@presight.ai
703,EMP0704,Omar Al Mansoori,NaN,omar.al.mansoori.emp0704@presight.ai
790,EMP0791,Anna Hernandez,NaN,anna.hernandez@presight.ai
849,EMP0850,Divya Mehta,NaN,divya.mehta@presight.ai
851,EMP0852,Mateo Reddy,NaN,mateo.reddy.emp0852@presight.ai


### Issue 2 — Invalid date formats
`hire_date` holds sentinel junk (`-999`, `99999-01-01`) that doesn't parse as `%Y-%m-%d`. These become `NULL` rather than being silently coerced or guessed.

In [25]:
invalid_date = pd.to_datetime(raw_employees["hire_date"], errors="coerce", format="%Y-%m-%d").isna()
print("rows affected:", int(invalid_date.sum()))
raw_employees.loc[invalid_date, "hire_date"].value_counts()

rows affected: 8


hire_date
-999           5
99999-01-01    3
Name: count, dtype: int64[pyarrow]

### Issue 3 — Implausible dates (+ recovery from salary history)
Parsable dates outside a plausible business range (before 1990, or in the future) are also set to `NULL`. For any employee whose hire date became `NULL` (issues 2 + 3 combined), the pipeline then tries to recover it from `employees_salary_history.csv`'s `Hire` record — a genuinely independent second source, not a guess. Rows that still have no valid date afterwards are flagged `active_without_hire_date_REVIEW` in `dq_flags` if their status is Active, rather than silently kept.

In [26]:
bad_date = invalid_date | (
    pd.to_datetime(raw_employees["hire_date"], errors="coerce", format="%Y-%m-%d").notna()
    & (
        (pd.to_datetime(raw_employees["hire_date"], errors="coerce", format="%Y-%m-%d") < HIRE_DATE_MIN)
        | (pd.to_datetime(raw_employees["hire_date"], errors="coerce", format="%Y-%m-%d") > pd.Timestamp.today())
    )
)
pd.DataFrame({
    "employee_id": raw_employees.loc[bad_date, "employee_id"],
    "raw_hire_date": raw_employees.loc[bad_date, "hire_date"],
    "recovered_hire_date": employees_clean.loc[bad_date, "hire_date"],
    "dq_flags": employees_clean.loc[bad_date, "dq_flags"],
})

,employee_id,raw_hire_date,recovered_hire_date,dq_flags
37,EMP0038,-999,NaT,hire_date_invalid;active_without_hire_date_REVIEW
159,EMP0160,99999-01-01,NaT,hire_date_invalid;active_without_hire_date_REVIEW
174,EMP0175,-999,NaT,hire_date_invalid;active_without_hire_date_REVIEW
394,EMP0395,99999-01-01,NaT,hire_date_invalid;active_without_hire_date_REVIEW
550,EMP0551,99999-01-01,NaT,hire_date_invalid;active_without_hire_date_REVIEW
599,EMP0600,-999,NaT,hire_date_invalid;active_without_hire_date_REVIEW
854,EMP0855,-999,NaT,hire_date_invalid;active_without_hire_date_REVIEW
972,EMP0973,-999,NaT,hire_date_invalid;active_without_hire_date_REVIEW


### Issue 4 — Numeric out-of-range
`years_experience < 0` is physically impossible. Fixed by imputing the **median** experience for that employee's level (robust to outliers, unlike the mean). Values that are merely *outside the plausible band for their level* (e.g. a Junior with 10 years) are flagged for review rather than silently changed, since that alone doesn't prove the value is wrong.

In [27]:
neg_exp = raw_employees["years_experience"] < 0
print("rows affected:", int(neg_exp.sum()))
pd.DataFrame({
    "employee_id": raw_employees.loc[neg_exp, "employee_id"],
    "level": raw_employees.loc[neg_exp, "level"],
    "raw_years_experience": raw_employees.loc[neg_exp, "years_experience"],
    "clean_years_experience (level median)": employees_clean.loc[neg_exp, "years_experience"],
})

rows affected: 5


,employee_id,level,raw_years_experience,clean_years_experience (level median)
215,EMP0216,Junior,-1,1
355,EMP0356,Junior,-1,1
444,EMP0445,Senior,-1,9
582,EMP0583,Senior,-1,9
719,EMP0720,Mid,-1,4


### Issue 5 — Logical inconsistency: salary vs level
3 Juniors are paid 66K-76K AED — about 5x the Junior band ceiling, which looks like a keying error (an extra digit / wrong multiplier) rather than genuine pay. The fix divides by 5 **only when that lands the value back inside the level's band**; if it still doesn't fit, the level median is used instead. `LEVEL_SALARY_BANDS` is the business-rule config driving this.

In [28]:
lo = raw_employees["level"].map({k: v[0] for k, v in LEVEL_SALARY_BANDS.items()})
hi = raw_employees["level"].map({k: v[1] for k, v in LEVEL_SALARY_BANDS.items()})
salary_off = (raw_employees["salary"] < lo) | (raw_employees["salary"] > hi)
print("rows affected:", int(salary_off.sum()))
pd.DataFrame({
    "employee_id": raw_employees.loc[salary_off, "employee_id"],
    "level": raw_employees.loc[salary_off, "level"],
    "raw_salary": raw_employees.loc[salary_off, "salary"],
    "raw_salary / 5": (raw_employees.loc[salary_off, "salary"] / 5).round(),
    "level_band": raw_employees.loc[salary_off, "level"].map(LEVEL_SALARY_BANDS),
    "clean_salary": employees_clean.loc[salary_off, "salary"],
})

rows affected: 3


,employee_id,level,raw_salary,raw_salary / 5,level_band,clean_salary
355,EMP0356,Junior,76507,15301.0,"(12000, 20000)",15301
866,EMP0867,Junior,66797,13359.0,"(12000, 20000)",13359
899,EMP0900,Junior,71212,14242.0,"(12000, 20000)",14242


### Issue 6 — Status conflicts / referential issues
Three sub-checks, all logged and flagged rather than silently fixed where the correct answer isn't obvious:
- **(a)** Active employees with no verifiable hire date (already shown above, flag `active_without_hire_date_REVIEW`).
- **(b)** `manager_id = 'EMP0000'` is a placeholder that doesn't exist as an employee — set to `NULL` (top of the hierarchy), not left as a dangling reference.
- **(c)** Active staff reporting to an `Inactive` manager — flagged for HR review (finds 0 rows on this dataset; kept anyway as a guardrail against future data).

In [29]:
orphan_mgr = raw_employees["manager_id"].notna() & ~raw_employees["manager_id"].isin(raw_employees["employee_id"])
print("(b) manager_id pointing at a non-existent employee:", int(orphan_mgr.sum()))
pd.DataFrame({
    "employee_id": raw_employees.loc[orphan_mgr, "employee_id"],
    "raw_manager_id": raw_employees.loc[orphan_mgr, "manager_id"],
    "clean_manager_id": employees_clean.loc[orphan_mgr, "manager_id"],
})

(b) manager_id pointing at a non-existent employee: 5


,employee_id,raw_manager_id,clean_manager_id
0,EMP0001,EMP0000,NaN
1,EMP0002,EMP0000,NaN
2,EMP0003,EMP0000,NaN
3,EMP0004,EMP0000,NaN
4,EMP0005,EMP0000,NaN


In [30]:
inactive_ids = raw_employees.loc[raw_employees["status"] == "Inactive", "employee_id"]
inactive_mgr = raw_employees["manager_id"].isin(inactive_ids) & (raw_employees["status"] == "Active")
print("(c) Active staff reporting to an Inactive manager:", int(inactive_mgr.sum()), "(0 is expected on this dataset)")

(c) Active staff reporting to an Inactive manager: 0 (0 is expected on this dataset)


### Every fix, all in one place
`dq_flags` is the per-row audit trail: each fix appends its own label, so any row's history is readable without cross-referencing five different masks.

In [31]:
pd.Series(RUN_STATS["quality_fixes"], name="rows_affected").rename_axis("fix").to_frame()

,rows_affected
fix,
projects: null budget -> 0,32
projects: null actual_cost -> 0,60
employees: missing employee_id (unfixable),0
employees: missing full_name (unfixable),0
employees: missing department (unfixable),0
employees: missing role (unfixable),0
employees: missing level (unfixable),0
employees: missing status (unfixable),0
employees: missing email -> derived from name,10


In [32]:
# Rows touched per fix type, read back from the audit column itself (not RUN_STATS) as a cross-check
employees_clean["dq_flags"].dropna().str.split(";").explode().value_counts()

dq_flags
                                   970
email_derived                       10
hire_date_invalid                    8
active_without_hire_date_REVIEW      8
manager_placeholder_nulled           5
experience_imputed                   5
salary_corrected                     3
Name: count, dtype: int64

## Recap
- `projects_clean` — 500 rows, 17 columns (11 raw + 4 derived metrics + `status_category` + `risk_level`)
- `employees_clean` — 1,000 rows, 13 columns (12 raw + `dq_flags`)
- Every fix above is vectorised (no `.iterrows()`) and logged with its row count via `logger` / `RUN_STATS`.
- Full run, including the JSON/TXT summary files: `python 01_foundations/etl_pipeline.py`.

In [33]:
print("projects_clean :", projects_clean.shape)
print("employees_clean:", employees_clean.shape)

projects_clean : (500, 17)
employees_clean: (1000, 13)
